<a href="https://colab.research.google.com/github/arshad831/zain_2026/blob/main/Zain_SQL_Agent_Minimal_Gradio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zain Jordan Minimal SQL Agent Gradio App

This notebook has one main runnable cell. It assumes the database is already uploaded to `/content/zain_customer_360_ai_demo.db`.

In [1]:
# Install packages
!pip install -q -U gradio pandas sqlalchemy langchain langchain-openai langchain-community openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 k

In [2]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


Saving zain_customer_360_ai_demo.db to zain_customer_360_ai_demo.db
Uploaded files: ['zain_customer_360_ai_demo.db']


In [ ]:


import os
import traceback
from pathlib import Path

import gradio as gr

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit


DB_PATH = "/content/zain_customer_360_ai_demo.db"
MODEL_NAME = "gpt-4.1-mini"

# Load OpenAI key from Colab Secrets if available
try:
    from google.colab import userdata
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass


SYSTEM_PROMPT = """
You are a telecom business intelligence SQL agent for Zain Jordan.

You are connected to a SQLite Customer 360 database.

Rules:
- Use SELECT queries only.
- Never modify the database.
- Inspect the schema before writing SQL.
- Limit results to 10 rows unless the user asks otherwise.
- Do not guess facts that are not in the database.
- Explain the result in simple business language.

Useful context:
- Churn analysis usually uses customers and customer_churn_scores.
- Revenue/value analysis usually uses customer_value_segments, invoices, payments, or transactions.
- Complaint analysis usually uses complaints and support_interactions.
- Campaign analysis usually uses campaigns and customer_campaign_responses.

Final answer format:
1. Direct Answer
2. Key Numbers
3. Business Interpretation
4. Recommended Next Action
""".strip()


agent = None


def get_sql_agent():
    global agent

    if agent is not None:
        return agent

    if not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("OPENAI_API_KEY is missing. Add it to Colab Secrets or environment variables.")

    if not Path(DB_PATH).exists():
        raise FileNotFoundError(f"Database not found at {DB_PATH}. Please upload it to Colab first.")

    db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

    llm = init_chat_model(
        MODEL_NAME,
        model_provider="openai"
    )

    toolkit = SQLDatabaseToolkit(
        db=db,
        llm=llm
    )

    tools = toolkit.get_tools()

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=SYSTEM_PROMPT
    )

    return agent


def run_sql_agent(question):
    try:
        if not question or not question.strip():
            return "Please enter a question."

        sql_agent = get_sql_agent()

        result = sql_agent.invoke({
            "messages": [
                {"role": "user", "content": question}
            ]
        })

        final_message = result["messages"][-1]

        if hasattr(final_message, "content"):
            return final_message.content

        return str(final_message)

    except Exception as e:
        return f"""
Error:

{str(e)}

Traceback:

{traceback.format_exc()}
"""


with gr.Blocks(title="Zain Jordan SQL Agent") as demo:

    gr.Markdown("# Zain Jordan Customer 360 SQL Agent")
    gr.Markdown("""
Ask business questions from the uploaded SQLite Customer 360 database.

Database path used by this app:

`/content/zain_customer_360_ai_demo.db`

Default model:

`gpt-4.1-mini`
""")

    question = gr.Textbox(
        label="Ask a business question",
        lines=4,
        value="Which cities have the most high-risk churn customers? Show the top 10 cities."
    )

    run_button = gr.Button("Run SQL Agent", variant="primary")

    answer = gr.Markdown(label="Answer")

    run_button.click(
        fn=run_sql_agent,
        inputs=question,
        outputs=answer
    )


demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d167b5808e69d5c80c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
